In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# GROW 晚窗控制：固定终态验证

选择 GPU 运行时后点击“全部运行”。本次固定 2 个内容 × 2 个种子 ×
OFF/A/B = 12 个视频生成轨迹，其中 8 个携带消息；仅检查终态潜变量，
不执行 VAE、MP4 或质量测量。失败保留在固定分母中，不自动补跑或扫描参数。

唯一方法变化是将原有 20 步控制从零基索引 10..29 移至 30..49。
保持 channel 0、全片正交 DCT-II、双轴频率 2..9、16 位消息、
每片每位 4 次重复 × 46 片、幅度 0.5、eta 0.1、CFG 5、50 步原生 UniPC。
读出使用 184 个系数符号票，零和平票为擦除；系数均值仅用于诊断。
每个受控步以同状态/历史的无控制 shadow 测量真实控制响应，shadow 不更新 live 历史。

固定预算：1200 次 Transformer 前向、600 次 live scheduler step、
160 次局部梯度、160 次 shadow step；无 Transformer 反传。
8 个携带消息的终态全部 16/16 正确且零擦除，才标记 media_eligible。
这仅表示可考虑后续媒体验证，不自动运行媒体阶段，不代表科学成功。
最后一步可使终态等于受控 predicted-clean，因此终态恢复不能单独证明
此前控制持续保留或 MP4 可读；与旧运行的因果比较还需核对环境与初始状态。

独立落盘目录：`MyDrive/Video-WM/GROWLateControl/grow_late_control_<UTC>`。
同目录父级保存 source.zip 和 launcher.log；运行目录保存 manifest/result，
各 case 保存 config、codebook、调度、张量及日志。最终摘要包含 8/8 gate。
依赖沿用先前 torch 2.11.0+cu128 / diffusers 0.40.0；匹配的 torch 不重装，
不新增 GPU 型号限制。当前交付仅静态核验，未运行模型或 GPU 实验。

固定源码 SHA：482222f5637fa3a38c170f9ee762fc03eb08ce2e。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = '482222f5637fa3a38c170f9ee762fc03eb08ce2e'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'grow_late_control_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/GROWLateControl') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.grow_late_control_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({'status': result['status'], 'video_denominator': result['video_denominator'], 'media_gate': result.get('media_gate'), 'fixed_calls': result['fixed_calls'], 'actual_calls_observed': result.get('actual_calls_observed'), 'cases': {k: v['status'] for k, v in result['cases'].items()}}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
